In [1]:
import duckdb
import os
import time

DATA_DIR    = os.path.expanduser('~/kkbox-churn/data/')
PARQUET_DIR = os.path.expanduser('~/kkbox-churn/data/parquet/')
os.makedirs(PARQUET_DIR, exist_ok=True)

con = duckdb.connect()
print("Output directory:", PARQUET_DIR)

Output directory: /Users/harshithnr/kkbox-churn/data/parquet/


In [3]:
small_files = {
    'train_v2'       : 'train_v2.csv',
    'members_v3'     : 'members_v3.csv',
    'transactions'   : 'transactions.csv',
    'transactions_v2': 'transactions_v2.csv',
    'user_logs_v2'   : 'user_logs_v2.csv',
}

for name, filename in small_files.items():
    src = DATA_DIR + filename
    dst = PARQUET_DIR + name + '.parquet'
    
    print(f"Converting {filename}...")
    start = time.time()
    
    con.execute(f"""
        COPY (SELECT * FROM read_csv_auto('{src}'))
        TO '{dst}'
        (FORMAT PARQUET, COMPRESSION ZSTD)
    """)
    
    elapsed = time.time() - start
    size = os.path.getsize(dst) / (1024**2)
    print(f"✅ Done — {size:.1f} MB — {elapsed:.1f}s\n")

print("All small files converted.")

Converting train_v2.csv...
✅ Done — 30.8 MB — 0.1s

Converting members_v3.csv...
✅ Done — 225.2 MB — 0.5s

Converting transactions.csv...
✅ Done — 726.7 MB — 1.4s

Converting transactions_v2.csv...
✅ Done — 50.3 MB — 0.2s

Converting user_logs_v2.csv...
✅ Done — 732.6 MB — 1.2s

All small files converted.


In [4]:
src = DATA_DIR + 'user_logs.csv'
dst = PARQUET_DIR + 'user_logs.parquet'

print("Converting user_logs.csv (28GB) — this will take 10-15 minutes...")
start = time.time()

con.execute(f"""
    COPY (SELECT * FROM read_csv_auto('{src}'))
    TO '{dst}'
    (FORMAT PARQUET, COMPRESSION ZSTD)
""")

elapsed = time.time() - start
size = os.path.getsize(dst) / (1024**3)
print(f"✅ Done — {size:.2f} GB — {elapsed:.1f}s")

Converting user_logs.csv (28GB) — this will take 10-15 minutes...
✅ Done — 6.15 GB — 19.8s


In [5]:
print("=== PARQUET FILE INVENTORY ===\n")

expected = {
    'train_v2'      : 970960,
    'members_v3'    : 6769473,
    'transactions'  : 21547746,
    'transactions_v2': 1431009,
    'user_logs_v2'  : 18396362,
    'user_logs'     : 392106543,
}

all_good = True
for name, expected_rows in expected.items():
    path = PARQUET_DIR + name + '.parquet'
    
    size = os.path.getsize(path) / (1024**2)
    count = con.execute(f"SELECT COUNT(*) FROM '{path}'").fetchone()[0]
    match = "✅" if count == expected_rows else "❌"
    
    print(f"{match} {name:<20} {count:>12,} rows   {size:>8.1f} MB")
    if count != expected_rows:
        all_good = False

print(f"\n{'All files verified correctly.' if all_good else 'MISMATCH DETECTED — check above.'}")
con.close()

=== PARQUET FILE INVENTORY ===

✅ train_v2                  970,960 rows       30.8 MB
✅ members_v3              6,769,473 rows      225.2 MB
✅ transactions           21,547,746 rows      726.7 MB
✅ transactions_v2         1,431,009 rows       50.3 MB
✅ user_logs_v2           18,396,362 rows      732.6 MB
✅ user_logs             392,106,543 rows     6296.2 MB

All files verified correctly.
